<a href="https://colab.research.google.com/github/doniyor117/olist-marketplace-analysis/blob/main/02-seller-quality/analysis_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 02 - Seller Quality

Olist doesn't choose what gets sold on its marketplace, but it chooses who
sells. Buyers see one Olist storefront, not separate shops, so a seller who
ships late or gets bad reviews drags down how the whole store looks.

This project builds a scorecard for each seller from review scores, shipping
times, and cancellations, then groups sellers by what Olist should do about
them. The questions below are starting assumptions, not conclusions.

This project is done mostly in SQL with DuckDB.

## Questions:

1. How does seller performance affect Olist, and in what measure?
2. What share of low review scores do the worst sellers account for?
3. What should Olist do with each seller: training, incentives, or removal?


In [1]:
! pip install -q duckdb jupysql

In [2]:
from pathlib import Path
import subprocess
import os

In [3]:
REPO_URL  = "https://github.com/doniyor117/olist-marketplace-analysis.git"
REPO_NAME = "olist-marketplace-analysis"

def find_root(start):
    for p in [start, *start.parents]:
        if (p / ".git").exists():
            return p
    return None

ROOT = find_root(Path.cwd())
if ROOT is None:
    if not (Path.cwd() / REPO_NAME).exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    ROOT = (Path.cwd() / REPO_NAME).resolve()

FIGURES_DIR = ROOT / "02-seller-quality" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

QUERIES_DIR = ROOT / "02-seller-quality" / "sql"
QUERIES_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

ROOT: /home/doniyor/Projects/portfolio-projects/olist-marketplace-analysis


## Downloading the dataset:

In [4]:
LOCAL = os.path.join(ROOT, "data", "raw")
KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce"

try:
    import kagglehub
    path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
except Exception:
    path = LOCAL
    if not os.path.exists(os.path.join(path, "olist_orders_dataset.csv")):
        os.makedirs(path, exist_ok=True)
        zp, _ = urllib.request.urlretrieve(KAGGLE_URL)
        zipfile.ZipFile(zp).extractall(path)

print("Path to ecommerce dataset files:", path)

Path to ecommerce dataset files: /home/doniyor/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


In [5]:
%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [6]:
tables = os.listdir(path)
tables

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [7]:
%%sql
CREATE TABLE IF NOT EXISTS orders          AS SELECT * FROM '{{path}}/olist_orders_dataset.csv';
CREATE TABLE IF NOT EXISTS order_items     AS SELECT * FROM '{{path}}/olist_order_items_dataset.csv';
CREATE TABLE IF NOT EXISTS order_payments  AS SELECT * FROM '{{path}}/olist_order_payments_dataset.csv';
CREATE TABLE IF NOT EXISTS order_reviews   AS SELECT * FROM '{{path}}/olist_order_reviews_dataset.csv';
CREATE TABLE IF NOT EXISTS sellers         AS SELECT * FROM '{{path}}/olist_sellers_dataset.csv';
CREATE TABLE IF NOT EXISTS customers       AS SELECT * FROM '{{path}}/olist_customers_dataset.csv';
CREATE TABLE IF NOT EXISTS products        AS SELECT * FROM '{{path}}/olist_products_dataset.csv';
CREATE TABLE IF NOT EXISTS category_translation AS SELECT * FROM '{{path}}/product_category_name_translation.csv';
CREATE TABLE IF NOT EXISTS geolocation     AS SELECT * FROM '{{path}}/olist_geolocation_dataset.csv';

Running query in 'duckdb:///:memory:'

Count


## Cleanining & Validation: